# Sandbox — smoke-test `lib.py` pieces

Throwaway notebook for poking at functions in `lib.py` without contaminating
the experiment notebooks. Use this to verify the hook returns the right
shape, the tokenizer behaves as expected, MM direction has a sane magnitude,
etc. Cells are independent — run whichever you need.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd()))

import numpy as np
import torch
import lib

## Tokenizer + prompt construction

In [2]:
# Load tokenizer only (no model — fast) to test prompt building.
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(lib.MODEL_PATH)
tokenizer.padding_side = "left"
if tokenizer.pad_token is None: tokenizer.pad_token = tokenizer.eos_token

p_honest = lib._make_prompt("an honest", "Paris is the capital of")
p_untrue = lib._make_prompt("an untruthful", "Paris is the capital of")
print("Honest prompt:")
print(repr(p_honest))
print()
print("Untruthful prompt (same prefix):")
print(repr(p_untrue))

# Tokenize both — verify last token is the same (it's the prefix end), and
# that left-padding works as expected.
batch = tokenizer([p_honest, p_untrue], padding=True, return_tensors="pt")
print(f"\ninput_ids shape: {batch['input_ids'].shape}")
print(f"Last token of each (decoded): {[tokenizer.decode([t]) for t in batch['input_ids'][:, -1]]}")
print(f"Last 5 tokens of honest:      {tokenizer.convert_ids_to_tokens(batch['input_ids'][0, -5:].tolist())}")
print(f"Last 5 tokens of untruthful:  {tokenizer.convert_ids_to_tokens(batch['input_ids'][1, -5:].tolist())}")

Honest prompt:
"<|im_start|>user\nPretend you're an honest person making statements about the world.<|im_end|>\n<|im_start|>assistant\nParis is the capital of"

Untruthful prompt (same prefix):
"<|im_start|>user\nPretend you're an untruthful person making statements about the world.<|im_end|>\n<|im_start|>assistant\nParis is the capital of"

input_ids shape: torch.Size([2, 29])
Last token of each (decoded): [' of', ' of']
Last 5 tokens of honest:      ['Paris', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof']
Last 5 tokens of untruthful:  ['Paris', 'Ġis', 'Ġthe', 'Ġcapital', 'Ġof']


## RepE pair construction on a tiny subset

In [3]:
train, test = lib.build_repe_pairs(
    lib.DATA_DIR / "facts_true_false.csv",
    tokenizer,
    seed=0,
    n_train_pairs=4,   # tiny — just to inspect output shape
    n_test_pairs=2,
)
print(f"Train rows: {len(train)}, Test rows: {len(test)}")
print(f"\nUnique statement_ids in train: {sorted({r['statement_id'] for r in train})}")
print(f"Label counts (train): honest={sum(r['label']==1 for r in train)}, "
      f"untruthful={sum(r['label']==0 for r in train)}")
print(f"\nFirst 4 prompts in train:")
for r in train[:4]:
    print(f"  [label={r['label']}, sid={r['statement_id']}]")
    print(f"    {r['prompt']!r}")

Train rows: 68, Test rows: 18

Unique statement_ids in train: [0, 1, 2, 3]
Label counts (train): honest=34, untruthful=34

First 4 prompts in train:
  [label=1, sid=2]
    "<|im_start|>user\nPretend you're an honest person making statements about the world.<|im_end|>\n<|im_start|>assistant\nThe four fundamental forces of"
  [label=0, sid=2]
    "<|im_start|>user\nPretend you're an untruthful person making statements about the world.<|im_end|>\n<|im_start|>assistant\nThe four fundamental forces of nature are"
  [label=1, sid=3]
    "<|im_start|>user\nPretend you're an honest person making statements about the world.<|im_end|>\n<|im_start|>assistant\nSound travels as"
  [label=0, sid=2]
    "<|im_start|>user\nPretend you're an untruthful person making statements about the world.<|im_end|>\n<|im_start|>assistant\nThe four fundamental forces of nature are gravity, electromagnetism, the strong nuclear force,"


## Model load (slow — only run when you need to test the hook)

In [4]:
model, tok = lib.load_model_and_tokenizer()
print(f"n_layers={model.config.num_hidden_layers}, hidden={model.config.hidden_size}")
print(f"GPU memory after load: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

n_layers=36, hidden=2560
GPU memory after load: 8.04 GB


## Smoke test: `collect_activations`

Tiny batch (4 prompts) → verify output shape, dtype, that vectors aren't
all-NaN, and that the same statement under honest vs untruthful gives
*different* activations (sanity: if the two are equal, you're extracting at
a position before the SP took effect, which means your hook is wrong).

In [6]:
# 2 statements x 2 personas = 4 prompts
prompts = [
    lib._make_prompt("an honest",     "The sun is a"),
    lib._make_prompt("an untruthful", "The sun is a"),
    lib._make_prompt("an honest",     "Water boils at"),
    lib._make_prompt("an untruthful", "Water boils at"),
]
labels = [1, 0, 1, 0]


acts = lib.collect_activations(model, tok, prompts, layer=14)

print(f"acts shape: {acts.shape}")  # expect (4, hidden)
print(f"acts dtype: {acts.dtype}")
print(f"any NaN:    {torch.isnan(acts).any().item()}")
print(f"|acts|:     {acts.float().norm(dim=-1).tolist()}")

# Sanity: same statement, different persona — activations should differ.
diff_sun   = (acts[0] - acts[1]).float().norm().item()
diff_water = (acts[2] - acts[3]).float().norm().item()
print(f"\nhonest vs untruthful |Δ| (sun):   {diff_sun:.3f}")
print(f"honest vs untruthful |Δ| (water): {diff_water:.3f}")
print("Both should be clearly > 0. If ~0, the hook is reading something prompt-shared.")



acts = lib.collect_activations_nnsight(model, tok, prompts, layer=14)

print(f"acts shape: {acts.shape}")  # expect (4, hidden)
print(f"acts dtype: {acts.dtype}")
print(f"any NaN:    {torch.isnan(acts).any().item()}")
print(f"|acts|:     {acts.float().norm(dim=-1).tolist()}")

# Sanity: same statement, different persona — activations should differ.
diff_sun   = (acts[0] - acts[1]).float().norm().item()
diff_water = (acts[2] - acts[3]).float().norm().item()
print(f"\nhonest vs untruthful |Δ| (sun):   {diff_sun:.3f}")
print(f"honest vs untruthful |Δ| (water): {diff_water:.3f}")
print("Both should be clearly > 0. If ~0, the hook is reading something prompt-shared.")

acts shape: torch.Size([4, 2560])
acts dtype: torch.bfloat16
any NaN:    False
|acts|:     [52.73270034790039, 53.740211486816406, 47.30195617675781, 48.39247512817383]

honest vs untruthful |Δ| (sun):   7.268
honest vs untruthful |Δ| (water): 8.774
Both should be clearly > 0. If ~0, the hook is reading something prompt-shared.
acts shape: torch.Size([4, 2560])
acts dtype: torch.bfloat16
any NaN:    False
|acts|:     [52.73270034790039, 53.740211486816406, 47.30195617675781, 48.39247512817383]

honest vs untruthful |Δ| (sun):   7.268
honest vs untruthful |Δ| (water): 8.774
Both should be clearly > 0. If ~0, the hook is reading something prompt-shared.


## Smoke test: `mm_direction`

In [7]:
# Dummy data: two well-separated Gaussians. MM direction should point along
# the separation axis (here: the all-ones direction).
rng = np.random.default_rng(0)
D = 64
X_neg = rng.normal(loc=-1.0, scale=1.0, size=(200, D))
X_pos = rng.normal(loc=+1.0, scale=1.0, size=(200, D))
X = np.concatenate([X_neg, X_pos], axis=0).astype(np.float32)
y = np.array([0]*200 + [1]*200)

d = lib.mm_direction(X, y)
print(f"d shape: {d.shape}")
print(f"d mean coord (expect ~2.0 since means differ by 2 per dim): {d.mean():.3f}")
print(f"d std across coords (expect small, since true direction is uniform): {d.std():.3f}")

# Project and check AUROC — should be near 1.0
scores = lib.mm_score(X, d)
print(f"MM AUROC on toy data: {lib.auroc(scores, y):.4f}")

d shape: (64,)
d mean coord (expect ~2.0 since means differ by 2 per dim): 2.003
d std across coords (expect small, since true direction is uniform): 0.098
MM AUROC on toy data: 1.0000


## Cache I/O smoke test

In [8]:
lib.save_cache("smoke", a=torch.tensor([1, 2, 3]), b=torch.randn(2, 3))
print("exists:", lib.cache_exists("smoke"))
blob = lib.load_cache("smoke")
print(blob)
lib.cache_path("smoke").unlink()

exists: True
{'a': tensor([1, 2, 3]), 'b': tensor([[-1.0973,  1.1886, -0.3139],
        [-0.6836, -0.6783, -0.6821]])}
